# CycleGAN: Joint GAN-A + GAN-B Training (Linux / Cloud GPU)

Trains both halves of the MIMIC pipeline end-to-end on **unpaired** healthy ↔ diseased leaf images.

## Generators
- **G_A** (diseased → healthy): UNet(3ch→3ch), warm-started from GAN-A Phase 1 checkpoint (first 3 input channels copied)
- **G_B** (healthy → diseased): UNet(3ch→3ch), trained from scratch

## Discriminators
- **D_A**: 70×70 PatchGAN — judges whether a leaf image looks healthy
- **D_B**: 70×70 PatchGAN — judges whether a leaf image looks diseased

## Loss Stack
- **Adversarial** (LSGAN): G_A fools D_A, G_B fools D_B
- **Cycle consistency**: L1 + SSIM + Perceptual on both forward and backward cycles
- **Identity**: G_A(healthy)≈healthy, G_B(diseased)≈diseased — prevents unnecessary color shifts

## Cycle Paths
```
Forward:  real_healthy  → G_B → fake_diseased → G_A → rec_healthy  ≈ real_healthy
Backward: real_diseased → G_A → fake_healthy  → G_B → rec_diseased ≈ real_diseased
```

## Data
- **Domain H** (healthy):  `data/gan_a/phase_1/healthy_target/`  (~380 clean healthy images)
- **Domain D** (diseased): `data/diseased/gls/`                   (~263 real GLS-infected images)

Unpaired — no correspondence between healthy and diseased images required.

**Target:** A100 GPU on Lambda Labs cloud

In [3]:
# 1. Imports
import os
import sys
import json
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
!{sys.executable} -m pip install tqdm
from tqdm import tqdm
import matplotlib
# Use inline backend in notebooks; use Agg only for non-notebook headless runs.
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 KB 3.2 MB/s eta 0:00:00


In [4]:
# 2. Import models and losses
from pathlib import Path
import importlib

# Resolve repo root (notebook is at src/gan/cyclegan_train_linux.ipynb)
repo_root = Path.cwd().resolve()
if repo_root.name == 'gan':
    repo_root = repo_root.parents[1]
elif repo_root.name == 'src':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Repo root: {repo_root}')

import src.models.unet_inpaint as unet_mod
import src.models.patchgan_discriminator as disc_mod
import src.models.losses as loss_mod
importlib.reload(unet_mod)
importlib.reload(disc_mod)
importlib.reload(loss_mod)

UNet = unet_mod.UNet
PatchGANDiscriminator = disc_mod.PatchGANDiscriminator
GANLoss = loss_mod.GANLoss
PerceptualLoss = loss_mod.PerceptualLoss
SSIMLoss = loss_mod.SSIMLoss
print('Generator, Discriminator, and loss modules loaded.')

Repo root: /lambda/nfs/test-2
Generator, Discriminator, and loss modules loaded.


In [5]:
# =============================================================
# 3. CONFIGURATION — edit these to switch datasets / hyperparams
# =============================================================

# ----- Data paths (relative to repo root) -----
# Domain H: clean healthy leaf images (targets from Phase 1 self-supervised training)
HEALTHY_IMAGE_DIR  = os.path.join(str(repo_root), 'data', 'healthy')
# Domain D: real GLS-diseased leaf images
DISEASED_IMAGE_DIR = os.path.join(str(repo_root), 'data', 'gls')

# ----- Train/test split -----
# 90/10 split applied per domain inside the dataset class (reproducible via seed)
TRAIN_SPLIT = 0.9
SPLIT_SEED  = 42

# ----- Output directories -----
CHECKPOINT_DIR = os.path.join(str(repo_root), 'src', 'models', 'checkpoints')
LOG_DIR        = os.path.join(str(repo_root), 'results', 'logs')
SAMPLE_DIR     = os.path.join(str(repo_root), 'results', 'figures', 'cyclegan_samples')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

# ----- Warm-start G_A from Phase 1 checkpoint -----
# Set to None to train both generators from scratch.
# Phase 1 checkpoint is 4-channel (RGB + outline → 3ch); we copy only the first 3
# input-channel weights and randomly initialize the 4th weight column.
PHASE1_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'full_ganA_epoch150_final.pth')

# ----- Resume full CycleGAN checkpoint (set to None to train from scratch) -----
RESUME_CHECKPOINT = None  # e.g. os.path.join(CHECKPOINT_DIR, 'cyclegan_epoch50.pth')

# ----- Hyperparameters (tuned for A100 80GB) -----
BATCH_SIZE    = 4      # CycleGAN doubles memory vs single GAN (4 generators + 4 discriminators active)
NUM_EPOCHS    = 200    # standard CycleGAN: 100 stable + 100 decay
N_EPOCHS_DECAY = 100   # epoch at which LR linear decay begins
IMAGE_SIZE    = 512
SAVE_EVERY    = 10
SAMPLE_EVERY  = 10
NUM_WORKERS   = 4
BUFFER_SIZE   = 50     # replay buffer size for discriminator stability

# ----- Generator -----
G_FEATURES      = [64, 128, 256, 512]
G_IN_CHANNELS   = 3    # 3ch RGB only (symmetric for both generators)
G_OUT_CHANNELS  = 3

# ----- Discriminator -----
D_BASE_FEATURES  = 64
D_N_LAYERS       = 3

# ----- Learning rates -----
G_LEARNING_RATE = 2e-4
D_LEARNING_RATE = 2e-4

# ----- Loss weights -----
LAMBDA_CYCLE      = 10.0   # L1 cycle consistency (forward + backward)
LAMBDA_SSIM       = 5.0    # SSIM cycle consistency
LAMBDA_PERCEPTUAL = 2.0    # VGG perceptual cycle consistency
LAMBDA_IDENTITY   = 5.0    # identity loss (prevents color shift)
LAMBDA_ADV        = 1.0    # adversarial weight
GAN_MODE          = 'lsgan'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB total')
print(f'Image size: {IMAGE_SIZE}x{IMAGE_SIZE}  |  Batch size: {BATCH_SIZE}')
print(f'Epochs: {NUM_EPOCHS} ({N_EPOCHS_DECAY} stable + {NUM_EPOCHS - N_EPOCHS_DECAY} LR decay)')
print(f'Loss weights — Cycle L1: {LAMBDA_CYCLE}  SSIM: {LAMBDA_SSIM}  '
      f'Perceptual: {LAMBDA_PERCEPTUAL}  Identity: {LAMBDA_IDENTITY}  Adv: {LAMBDA_ADV}')

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 42.4 GB total
Image size: 512x512  |  Batch size: 4
Epochs: 200 (100 stable + 100 LR decay)
Loss weights — Cycle L1: 10.0  SSIM: 5.0  Perceptual: 2.0  Identity: 5.0  Adv: 1.0


In [6]:
# 4. Dataset + utilities

class UnpairedLeafDataset(Dataset):
    """Loads images from a single domain (healthy OR diseased).
    Returns a 3-channel RGB tensor resized to IMAGE_SIZE.
    Datasets for the two domains are instantiated separately and iterated in parallel."""

    EXTS = ('.png', '.jpg', '.jpeg')

    def __init__(self, image_dir, image_size=512, split='train',
                 train_fraction=0.9, seed=42):
        all_files = sorted(
            f for f in os.listdir(image_dir)
            if f.lower().endswith(self.EXTS)
        )
        assert len(all_files) > 0, f'No images found in {image_dir}'

        rng = random.Random(seed)
        shuffled = all_files[:]
        rng.shuffle(shuffled)
        n_train = int(len(shuffled) * train_fraction)
        self.files = shuffled[:n_train] if split == 'train' else shuffled[n_train:]
        self.image_dir = image_dir

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),   # [0, 1]
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        img = Image.open(os.path.join(self.image_dir, self.files[index])).convert('RGB')
        return self.transform(img), self.files[index]


class ImageBuffer:
    """Replay buffer for discriminator training stability (CycleGAN standard trick).
    Randomly returns either the current batch or a stored past batch (50/50)."""

    def __init__(self, max_size=50):
        self.max_size = max_size
        self.images = []

    def push_and_pop(self, images):
        """images: [B, C, H, W] detached tensor. Returns same-shape tensor from buffer."""
        result = []
        for img in images.detach().unbind(0):  # iterate over batch
            if len(self.images) < self.max_size:
                self.images.append(img)
                result.append(img)
            elif random.random() > 0.5:
                # Swap a random stored image with the current one
                idx = random.randint(0, self.max_size - 1)
                stored = self.images[idx].clone()
                self.images[idx] = img
                result.append(stored)
            else:
                result.append(img)
        return torch.stack(result)


# Create train/test datasets for both domains
train_H = UnpairedLeafDataset(HEALTHY_IMAGE_DIR,  IMAGE_SIZE, 'train', TRAIN_SPLIT, SPLIT_SEED)
train_D = UnpairedLeafDataset(DISEASED_IMAGE_DIR, IMAGE_SIZE, 'train', TRAIN_SPLIT, SPLIT_SEED)
test_H  = UnpairedLeafDataset(HEALTHY_IMAGE_DIR,  IMAGE_SIZE, 'test',  TRAIN_SPLIT, SPLIT_SEED)
test_D  = UnpairedLeafDataset(DISEASED_IMAGE_DIR, IMAGE_SIZE, 'test',  TRAIN_SPLIT, SPLIT_SEED)

loader_H = DataLoader(train_H, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
loader_D = DataLoader(train_D, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

# Replay buffers (one per domain output)
buffer_A = ImageBuffer(BUFFER_SIZE)   # stores fake_healthy  (output of G_A)
buffer_B = ImageBuffer(BUFFER_SIZE)   # stores fake_diseased (output of G_B)

print(f'Domain H — train: {len(train_H)}, test: {len(test_H)}')
print(f'Domain D — train: {len(train_D)}, test: {len(test_D)}')
print(f'Batches per epoch: {min(len(loader_H), len(loader_D))} (shorter domain defines epoch length)')

Domain H — train: 17, test: 2
Domain D — train: 313, test: 35
Batches per epoch: 4 (shorter domain defines epoch length)


In [7]:
# 5. Networks, optimizers, scalers, schedulers

# --- Generators ---
# G_A: diseased (3ch) → healthy (3ch)   [GAN-A role]
# G_B: healthy  (3ch) → diseased (3ch)  [GAN-B role]
netG_A = UNet(in_channels=G_IN_CHANNELS, out_channels=G_OUT_CHANNELS, features=G_FEATURES).to(device)
netG_B = UNet(in_channels=G_IN_CHANNELS, out_channels=G_OUT_CHANNELS, features=G_FEATURES).to(device)

# --- Discriminators (unconditional: judge images in their own domain) ---
# D_A: judges healthy images  (real healthy vs fake healthy from G_A)
# D_B: judges diseased images (real diseased vs fake diseased from G_B)
netD_A = PatchGANDiscriminator(in_channels=3, base_features=D_BASE_FEATURES, n_layers=D_N_LAYERS).to(device)
netD_B = PatchGANDiscriminator(in_channels=3, base_features=D_BASE_FEATURES, n_layers=D_N_LAYERS).to(device)

# --- Warm-start G_A from GAN-A Phase 1 checkpoint ---
# Phase 1 used 4-channel input (RGB + leaf outline). We copy the first 3 channels
# of the first conv layer and randomly initialize the rest of that layer.
if PHASE1_CHECKPOINT and os.path.exists(PHASE1_CHECKPOINT) and RESUME_CHECKPOINT is None:
    ckpt = torch.load(PHASE1_CHECKPOINT, map_location=device, weights_only=False)
    phase1_state = ckpt.get('netG_state_dict', ckpt)

    # Adapt weights: Phase 1 first conv is [out_ch, 4, k, k]; our G_A is [out_ch, 3, k, k]
    adapted_state = {}
    cyclegan_state = netG_A.state_dict()
    for k, v in phase1_state.items():
        if k in cyclegan_state:
            if v.shape == cyclegan_state[k].shape:
                adapted_state[k] = v          # exact match — copy directly
            elif k.endswith('.weight') and v.dim() == 4 and v.shape[1] > G_IN_CHANNELS:
                # First conv: [out_ch, 4, k, k] → [out_ch, 3, k, k] (drop outline channel)
                adapted_state[k] = v[:, :G_IN_CHANNELS, :, :]
                print(f'  Adapted {k}: {tuple(v.shape)} → {tuple(adapted_state[k].shape)}')
            else:
                print(f'  Skipped {k}: shape mismatch {tuple(v.shape)} vs {tuple(cyclegan_state[k].shape)}')
        else:
            print(f'  Skipped {k}: key not in CycleGAN G_A')

    missing, unexpected = netG_A.load_state_dict(adapted_state, strict=False)
    print(f'G_A warm-started from {os.path.basename(PHASE1_CHECKPOINT)}')
    print(f'  Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}')
    del ckpt, phase1_state, adapted_state
    torch.cuda.empty_cache()
else:
    print('G_A initialised from scratch (no Phase 1 checkpoint loaded).')

# --- Optimizers (Adam with β1=0.5 — GAN standard) ---
optG = optim.Adam(
    list(netG_A.parameters()) + list(netG_B.parameters()),
    lr=G_LEARNING_RATE, betas=(0.5, 0.999)
)
optD_A = optim.Adam(netD_A.parameters(), lr=D_LEARNING_RATE, betas=(0.5, 0.999))
optD_B = optim.Adam(netD_B.parameters(), lr=D_LEARNING_RATE, betas=(0.5, 0.999))

# --- LR schedulers: constant for first N_EPOCHS_DECAY, then linear decay to 0 ---
def make_lr_lambda(start_decay_epoch, total_epochs):
    def lr_lambda(epoch):
        if epoch < start_decay_epoch:
            return 1.0
        return max(0.0, 1.0 - (epoch - start_decay_epoch) / (total_epochs - start_decay_epoch))
    return lr_lambda

lr_lambda = make_lr_lambda(N_EPOCHS_DECAY, NUM_EPOCHS)
sched_G   = optim.lr_scheduler.LambdaLR(optG,   lr_lambda=lr_lambda)
sched_D_A = optim.lr_scheduler.LambdaLR(optD_A, lr_lambda=lr_lambda)
sched_D_B = optim.lr_scheduler.LambdaLR(optD_B, lr_lambda=lr_lambda)

# --- Mixed precision scalers (one per optimizer) ---
scaler_G   = GradScaler('cuda')
scaler_D_A = GradScaler('cuda')
scaler_D_B = GradScaler('cuda')

# --- Loss functions ---
criterion_gan  = GANLoss(mode=GAN_MODE).to(device)
criterion_l1   = nn.L1Loss()
criterion_ssim = SSIMLoss().to(device)
criterion_perc = PerceptualLoss().to(device)

# --- Resume from checkpoint ---
START_EPOCH = 1
history = {
    'g_loss': [], 'd_a_loss': [], 'd_b_loss': [],
    'cycle_a_l1': [], 'cycle_b_l1': [],
    'cycle_a_ssim': [], 'cycle_b_ssim': [],
    'cycle_a_perc': [], 'cycle_b_perc': [],
    'idt_a': [], 'idt_b': [],
    'adv_a': [], 'adv_b': [],
    'epoch_time': [],
}
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    netG_A.load_state_dict(ckpt['netG_A_state_dict'])
    netG_B.load_state_dict(ckpt['netG_B_state_dict'])
    netD_A.load_state_dict(ckpt['netD_A_state_dict'])
    netD_B.load_state_dict(ckpt['netD_B_state_dict'])
    optG.load_state_dict(ckpt['optG_state_dict'])
    optD_A.load_state_dict(ckpt['optD_A_state_dict'])
    optD_B.load_state_dict(ckpt['optD_B_state_dict'])
    sched_G.load_state_dict(ckpt['sched_G_state_dict'])
    sched_D_A.load_state_dict(ckpt['sched_D_A_state_dict'])
    sched_D_B.load_state_dict(ckpt['sched_D_B_state_dict'])
    START_EPOCH = ckpt['epoch'] + 1
    if 'history' in ckpt:
        history = ckpt['history']
    print(f"Resumed from epoch {ckpt['epoch']} → continuing from {START_EPOCH}")
    del ckpt
    torch.cuda.empty_cache()
else:
    print('Starting fresh CycleGAN training.')

# --- Print model sizes ---
ga_params = sum(p.numel() for p in netG_A.parameters() if p.requires_grad)
gb_params = sum(p.numel() for p in netG_B.parameters() if p.requires_grad)
da_params = sum(p.numel() for p in netD_A.parameters() if p.requires_grad)
db_params = sum(p.numel() for p in netD_B.parameters() if p.requires_grad)
print(f'G_A params: {ga_params:,}  |  G_B params: {gb_params:,}')
print(f'D_A params: {da_params:,}  |  D_B params: {db_params:,}')
print(f'Total trainable: {ga_params + gb_params + da_params + db_params:,}')

# Sanity-check forward passes
with torch.no_grad():
    dummy_H = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    dummy_D = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    fake_d  = netG_B(dummy_H)          # H → D
    fake_h  = netG_A(dummy_D)          # D → H
    rec_h   = netG_A(fake_d)           # fake_D → rec_H (forward cycle)
    rec_d   = netG_B(fake_h)           # fake_H → rec_D (backward cycle)
    d_a_out = netD_A(fake_h)           # discriminate healthy
    d_b_out = netD_B(fake_d)           # discriminate diseased
print(f'G_A output: {tuple(fake_h.shape)}  |  G_B output: {tuple(fake_d.shape)}')
print(f'Cycle rec_H: {tuple(rec_h.shape)}  |  Cycle rec_D: {tuple(rec_d.shape)}')
print(f'D_A patch map: {tuple(d_a_out.shape)}  |  D_B patch map: {tuple(d_b_out.shape)}')
del dummy_H, dummy_D, fake_d, fake_h, rec_h, rec_d, d_a_out, d_b_out
torch.cuda.empty_cache()

G_A initialised from scratch (no Phase 1 checkpoint loaded).


3.5%

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/ubuntu/.cache/torch/hub/checkpoints/vgg16-397923af.pth


71.0%IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Starting fresh CycleGAN training.
G_A params: 31,043,651  |  G_B params: 31,043,651
D_A params: 2,764,737  |  D_B params: 2,764,737
Total trainable: 67,616,776
G_A output: (1, 3, 512, 512)  |  G_B output: (1, 3, 512, 512)
Cycle rec_H: (1, 3, 512, 512)  |  Cycle rec_D: (1, 3, 512, 512)
D_A patch map: (1, 1, 62, 62)  |  D_B patch map: (1, 1, 62, 62)


In [19]:
# 6. Training loop

def save_cycle_samples(netG_A, netG_B, dataset_H, dataset_D, device, epoch, sample_dir, num_samples=3):
    """Save a grid showing: real_H | fake_D (G_B) | rec_H (cycle) | real_D | fake_H (G_A) | rec_D (cycle)"""
    netG_A.eval()
    netG_B.eval()

    n = min(num_samples, len(dataset_H), len(dataset_D))
    fig, axs = plt.subplots(n, 6, figsize=(24, 5 * n))
    if n == 1:
        axs = axs[np.newaxis, :]  # ensure 2-D

    col_titles = ['Real Healthy', 'Fake Diseased (G_B)', 'Rec Healthy (cycle)',
                  'Real Diseased', 'Fake Healthy (G_A)', 'Rec Diseased (cycle)']
    for j, title in enumerate(col_titles):
        axs[0, j].set_title(title, fontsize=11, fontweight='bold')

    for i in range(n):
        real_h, fname_h = dataset_H[i]
        real_d, fname_d = dataset_D[i]
        real_h = real_h.unsqueeze(0).to(device)
        real_d = real_d.unsqueeze(0).to(device)

        with torch.no_grad():
            fake_d = netG_B(real_h)     # H → D
            fake_h = netG_A(real_d)     # D → H
            rec_h  = netG_A(fake_d)     # fake_D → rec_H
            rec_d  = netG_B(fake_h)     # fake_H → rec_D

        def to_np(t):
            return t.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()

        imgs = [to_np(real_h), to_np(fake_d), to_np(rec_h),
                to_np(real_d), to_np(fake_h), to_np(rec_d)]
        for j, im in enumerate(imgs):
            axs[i, j].imshow(im)
            axs[i, j].axis('off')
        axs[i, 0].set_ylabel(fname_h[:20], fontsize=7)
        axs[i, 3].set_ylabel(fname_d[:20], fontsize=7)
        del fake_d, fake_h, rec_h, rec_d

    plt.suptitle(f'CycleGAN — Epoch {epoch}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(sample_dir, f'samples_epoch{epoch:04d}.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    netG_A.train()
    netG_B.train()


print(f'Training from epoch {START_EPOCH} to {NUM_EPOCHS}')
print('=' * 80)

for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    epoch_start = time.time()
    netG_A.train(); netG_B.train()
    netD_A.train(); netD_B.train()

    # Metric accumulators
    run = {k: 0.0 for k in [
        'g', 'd_a', 'd_b',
        'cycle_a_l1', 'cycle_b_l1',
        'cycle_a_ssim', 'cycle_b_ssim',
        'cycle_a_perc', 'cycle_b_perc',
        'idt_a', 'idt_b', 'adv_a', 'adv_b',
    ]}
    n_samples = 0

    # Zip loaders — epoch length = shorter domain
    iter_H = iter(loader_H)
    iter_D = iter(loader_D)
    n_batches = min(len(loader_H), len(loader_D))

    progress = tqdm(range(n_batches), desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for _ in progress:
        real_H, _ = next(iter_H)   # [B, 3, H, W]  healthy
        real_D, _ = next(iter_D)   # [B, 3, H, W]  diseased
        bs = real_H.size(0)
        real_H = real_H.to(device, non_blocking=True)
        real_D = real_D.to(device, non_blocking=True)

        # ================================================================
        # Train D_A  (discriminates healthy images)
        # ================================================================
        optD_A.zero_grad()
        with autocast('cuda'):
            fake_H = netG_A(real_D).detach()   # G_A: diseased → fake_healthy (detached)
            pred_real = netD_A(real_H)
            loss_d_a_real = criterion_gan(pred_real, target_is_real=True)
            fake_H_buf = buffer_A.push_and_pop(fake_H)
            pred_fake = netD_A(fake_H_buf)
            loss_d_a_fake = criterion_gan(pred_fake, target_is_real=False)
            loss_D_A = 0.5 * (loss_d_a_real + loss_d_a_fake)

        scaler_D_A.scale(loss_D_A).backward()
        scaler_D_A.step(optD_A)
        scaler_D_A.update()

        # ================================================================
        # Train D_B  (discriminates diseased images)
        # ================================================================
        optD_B.zero_grad()
        with autocast('cuda'):
            fake_D = netG_B(real_H).detach()   # G_B: healthy → fake_diseased (detached)
            pred_real = netD_B(real_D)
            loss_d_b_real = criterion_gan(pred_real, target_is_real=True)
            fake_D_buf = buffer_B.push_and_pop(fake_D)
            pred_fake = netD_B(fake_D_buf)
            loss_d_b_fake = criterion_gan(pred_fake, target_is_real=False)
            loss_D_B = 0.5 * (loss_d_b_real + loss_d_b_fake)

        scaler_D_B.scale(loss_D_B).backward()
        scaler_D_B.step(optD_B)
        scaler_D_B.update()

        # ================================================================
        # Train G_A + G_B jointly
        # ================================================================
        optG.zero_grad()
        with autocast('cuda'):
            fake_D = netG_B(real_H)   # G_B: healthy  → fake_diseased
            fake_H = netG_A(real_D)   # G_A: diseased → fake_healthy

            # -- Adversarial --
            loss_adv_A = criterion_gan(netD_A(fake_H), target_is_real=True)   # G_A fools D_A
            loss_adv_B = criterion_gan(netD_B(fake_D), target_is_real=True)   # G_B fools D_B

            # -- Cycle consistency --
            # Forward cycle:  H → fake_D → rec_H  ≈ real_H
            rec_H = netG_A(fake_D)
            loss_cycle_A_l1   = LAMBDA_CYCLE      * criterion_l1(rec_H, real_H)
            loss_cycle_A_ssim = LAMBDA_SSIM       * criterion_ssim(rec_H, real_H)
            loss_cycle_A_perc = LAMBDA_PERCEPTUAL * criterion_perc(rec_H, real_H)

            # Backward cycle: D → fake_H → rec_D  ≈ real_D
            rec_D = netG_B(fake_H)
            loss_cycle_B_l1   = LAMBDA_CYCLE      * criterion_l1(rec_D, real_D)
            loss_cycle_B_ssim = LAMBDA_SSIM       * criterion_ssim(rec_D, real_D)
            loss_cycle_B_perc = LAMBDA_PERCEPTUAL * criterion_perc(rec_D, real_D)

            # -- Identity loss --
            # G_A(real_H) ≈ real_H  (should be near-identity on its target domain)
            # G_B(real_D) ≈ real_D  (should be near-identity on its target domain)
            idt_A = netG_A(real_H)
            idt_B = netG_B(real_D)
            loss_idt_A = LAMBDA_IDENTITY * criterion_l1(idt_A, real_H)
            loss_idt_B = LAMBDA_IDENTITY * criterion_l1(idt_B, real_D)

            # -- Total generator loss --
            loss_G = (
                LAMBDA_ADV * (loss_adv_A + loss_adv_B)
                + loss_cycle_A_l1   + loss_cycle_B_l1
                + loss_cycle_A_ssim + loss_cycle_B_ssim
                + loss_cycle_A_perc + loss_cycle_B_perc
                + loss_idt_A + loss_idt_B
            )

        scaler_G.scale(loss_G).backward()
        scaler_G.step(optG)
        scaler_G.update()

        # -- Accumulate metrics (weighted by batch size) --
        run['g']            += loss_G.item() * bs
        run['d_a']          += loss_D_A.item() * bs
        run['d_b']          += loss_D_B.item() * bs
        run['cycle_a_l1']   += loss_cycle_A_l1.item() * bs
        run['cycle_b_l1']   += loss_cycle_B_l1.item() * bs
        run['cycle_a_ssim'] += loss_cycle_A_ssim.item() * bs
        run['cycle_b_ssim'] += loss_cycle_B_ssim.item() * bs
        run['cycle_a_perc'] += loss_cycle_A_perc.item() * bs
        run['cycle_b_perc'] += loss_cycle_B_perc.item() * bs
        run['idt_a']        += loss_idt_A.item() * bs
        run['idt_b']        += loss_idt_B.item() * bs
        run['adv_a']        += loss_adv_A.item() * bs
        run['adv_b']        += loss_adv_B.item() * bs
        n_samples += bs

        progress.set_postfix({
            'G':     f'{loss_G.item():.3f}',
            'D_A':   f'{loss_D_A.item():.3f}',
            'D_B':   f'{loss_D_B.item():.3f}',
            'cyc_A': f'{(loss_cycle_A_l1 + loss_cycle_A_ssim).item():.4f}',
            'cyc_B': f'{(loss_cycle_B_l1 + loss_cycle_B_ssim).item():.4f}',
        })

        del fake_D, fake_H, rec_H, rec_D, idt_A, idt_B

    # -- LR scheduler step --
    sched_G.step()
    sched_D_A.step()
    sched_D_B.step()

    # -- Epoch averages --
    avg = {k: v / n_samples for k, v in run.items()}
    epoch_time = time.time() - epoch_start

    history['g_loss'].append(avg['g'])
    history['d_a_loss'].append(avg['d_a'])
    history['d_b_loss'].append(avg['d_b'])
    history['cycle_a_l1'].append(avg['cycle_a_l1'])
    history['cycle_b_l1'].append(avg['cycle_b_l1'])
    history['cycle_a_ssim'].append(avg['cycle_a_ssim'])
    history['cycle_b_ssim'].append(avg['cycle_b_ssim'])
    history['cycle_a_perc'].append(avg['cycle_a_perc'])
    history['cycle_b_perc'].append(avg['cycle_b_perc'])
    history['idt_a'].append(avg['idt_a'])
    history['idt_b'].append(avg['idt_b'])
    history['adv_a'].append(avg['adv_a'])
    history['adv_b'].append(avg['adv_b'])
    history['epoch_time'].append(epoch_time)

    lr_now = sched_G.get_last_lr()[0]
    print(f'Epoch {epoch}/{NUM_EPOCHS} [{epoch_time:.1f}s | lr={lr_now:.2e}] | '
          f'G: {avg["g"]:.4f}  D_A: {avg["d_a"]:.4f}  D_B: {avg["d_b"]:.4f}  '
          f'CycA_L1: {avg["cycle_a_l1"]:.4f}  CycB_L1: {avg["cycle_b_l1"]:.4f}  '
          f'IdtA: {avg["idt_a"]:.4f}  IdtB: {avg["idt_b"]:.4f}')

    # -- Save checkpoint --
    if epoch % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'cyclegan_epoch{epoch}.pth')
        torch.save({
            'epoch': epoch,
            'netG_A_state_dict': netG_A.state_dict(),
            'netG_B_state_dict': netG_B.state_dict(),
            'netD_A_state_dict': netD_A.state_dict(),
            'netD_B_state_dict': netD_B.state_dict(),
            'optG_state_dict':   optG.state_dict(),
            'optD_A_state_dict': optD_A.state_dict(),
            'optD_B_state_dict': optD_B.state_dict(),
            'sched_G_state_dict':   sched_G.state_dict(),
            'sched_D_A_state_dict': sched_D_A.state_dict(),
            'sched_D_B_state_dict': sched_D_B.state_dict(),
            'g_loss':   avg['g'],
            'd_a_loss': avg['d_a'],
            'd_b_loss': avg['d_b'],
            'history': history,
        }, ckpt_path)
        print(f'  >> Checkpoint saved: {ckpt_path}')

    # -- Save visual samples --
    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        save_cycle_samples(netG_A, netG_B, test_H, test_D, device, epoch, SAMPLE_DIR)
        print(f'  >> Samples saved to {SAMPLE_DIR}')

    torch.cuda.empty_cache()

# -- Save final model --
final_path = os.path.join(CHECKPOINT_DIR, f'cyclegan_epoch{NUM_EPOCHS}_final.pth')
torch.save({
    'epoch': NUM_EPOCHS,
    'netG_A_state_dict': netG_A.state_dict(),
    'netG_B_state_dict': netG_B.state_dict(),
    'netD_A_state_dict': netD_A.state_dict(),
    'netD_B_state_dict': netD_B.state_dict(),
    'g_loss':   history['g_loss'][-1],
    'd_a_loss': history['d_a_loss'][-1],
    'd_b_loss': history['d_b_loss'][-1],
    'history': history,
}, final_path)
print(f'Final model saved: {final_path}')

log_path = os.path.join(LOG_DIR, 'cyclegan_history.json')
with open(log_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved: {log_path}')

del optG, optD_A, optD_B, scaler_G, scaler_D_A, scaler_D_B
torch.cuda.empty_cache()
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Training from epoch 1 to 200


Epoch 1/200: 100%|██████████| 4/4 [00:02<00:00,  1.72it/s, G=59.632, D_A=11.849, D_B=13.605, cyc_A=8.3510, cyc_B=8.4623]


Epoch 1/200 [2.8s | lr=2.00e-04] | G: 52.7683  D_A: 8.4911  D_B: 6.5465  CycA_L1: 3.3301  CycB_L1: 3.4080  IdtA: 1.6850  IdtB: 1.7179
  >> Samples saved to /lambda/nfs/test-2/results/figures/cyclegan_samples


Epoch 2/200: 100%|██████████| 4/4 [00:02<00:00,  1.73it/s, G=33.857, D_A=4.239, D_B=3.404, cyc_A=8.2887, cyc_B=8.0111]  

Epoch 2/200 [2.8s | lr=2.00e-04] | G: 44.3310  D_A: 9.2334  D_B: 9.2795  CycA_L1: 3.3228  CycB_L1: 3.4114  IdtA: 1.6907  IdtB: 1.7162


...

Epoch 198/200 [2.8s | lr=4.00e-06] | G: 7.1103  D_A: 0.0148  D_B: 0.0544  CycA_L1: 0.9701  CycB_L1: 0.3218  IdtA: 0.3761  IdtB: 0.2006



Epoch 199/200: 100%|██████████| 4/4 [00:02<00:00,  1.73it/s, G=7.139, D_A=0.030, D_B=0.040, cyc_A=1.6144, cyc_B=0.1394]

Epoch 199/200 [2.8s | lr=2.00e-06] | G: 7.2029  D_A: 0.0223  D_B: 0.0355  CycA_L1: 1.0256  CycB_L1: 0.3396  IdtA: 0.3735  IdtB: 0.1950



Epoch 200/200: 100%|██████████| 4/4 [00:02<00:00,  1.73it/s, G=7.986, D_A=0.007, D_B=0.031, cyc_A=2.0339, cyc_B=0.4255]


Epoch 200/200 [2.8s | lr=0.00e+00] | G: 7.1395  D_A: 0.0317  D_B: 0.0365  CycA_L1: 1.0089  CycB_L1: 0.2671  IdtA: 0.3755  IdtB: 0.1766
  >> Checkpoint saved: /lambda/nfs/test-2/src/models/checkpoints/cyclegan_epoch200.pth
  >> Samples saved to /lambda/nfs/test-2/results/figures/cyclegan_samples
Final model saved: /lambda/nfs/test-2/src/models/checkpoints/cyclegan_epoch200_final.pth
Training history saved: /lambda/nfs/test-2/results/logs/cyclegan_history.json
GPU memory after cleanup: 1.66 GB


In [20]:
# 7. Plot training metrics

epochs = range(1, len(history['g_loss']) + 1)

fig, axs = plt.subplots(3, 3, figsize=(18, 14))

def plot(ax, key, label, color):
    ax.plot(epochs, history[key], color=color, linewidth=1.5)
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.grid(True)

plot(axs[0, 0], 'g_loss',       'Total Generator Loss',        'blue')
plot(axs[0, 1], 'd_a_loss',     'D_A Loss (healthy disc.)',     'red')
plot(axs[0, 2], 'd_b_loss',     'D_B Loss (diseased disc.)',    'darkred')

plot(axs[1, 0], 'cycle_a_l1',   'Forward Cycle L1 (H→D→H)',    'green')
plot(axs[1, 1], 'cycle_b_l1',   'Backward Cycle L1 (D→H→D)',   'darkgreen')
plot(axs[1, 2], 'cycle_a_ssim', 'Forward Cycle SSIM loss',      'purple')

plot(axs[2, 0], 'cycle_b_ssim', 'Backward Cycle SSIM loss',     'mediumpurple')
plot(axs[2, 1], 'idt_a',        'Identity Loss G_A (H→H)',      'orange')
plot(axs[2, 2], 'idt_b',        'Identity Loss G_B (D→D)',      'darkorange')

plt.suptitle('CycleGAN Training Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(LOG_DIR, 'cyclegan_loss_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Loss curves saved: {fig_path}')

# G vs D balance
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(epochs, history['g_loss'],   'b-',  label='G Loss (total)', linewidth=1.5)
ax2.plot(epochs, history['d_a_loss'], 'r-',  label='D_A Loss',       linewidth=1.5)
ax2.plot(epochs, history['d_b_loss'], 'r--', label='D_B Loss',       linewidth=1.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('Generator vs Discriminator Loss Balance')
ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'cyclegan_gd_balance.png'), dpi=150, bbox_inches='tight')
plt.show()

# Cycle loss comparison
fig3, ax3 = plt.subplots(figsize=(10, 4))
cyc_a = [a + b for a, b in zip(history['cycle_a_l1'], history['cycle_a_ssim'])]
cyc_b = [a + b for a, b in zip(history['cycle_b_l1'], history['cycle_b_ssim'])]
ax3.plot(epochs, cyc_a, 'g-',  label='Forward cycle (H→D→H)',  linewidth=1.5)
ax3.plot(epochs, cyc_b, 'b--', label='Backward cycle (D→H→D)', linewidth=1.5)
ax3.set_xlabel('Epoch'); ax3.set_ylabel('L1 + SSIM cycle loss')
ax3.set_title('Cycle Consistency Loss (L1 + SSIM)')
ax3.legend(); ax3.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'cyclegan_cycle_loss.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal metrics (epoch {len(history['g_loss'])})")
for k in ['g_loss', 'd_a_loss', 'd_b_loss', 'cycle_a_l1', 'cycle_b_l1', 'idt_a', 'idt_b']:
    print(f'  {k:20s}: {history[k][-1]:.6f}')
total_time = sum(history['epoch_time'])
print(f'  Total training time: {total_time/3600:.2f} hours')

Loss curves saved: /lambda/nfs/test-2/results/logs/cyclegan_loss_curves.png

Final metrics (epoch 200)
  g_loss              : 7.139544
  d_a_loss            : 0.031663
  d_b_loss            : 0.036456
  cycle_a_l1          : 1.008892
  cycle_b_l1          : 0.267102
  idt_a               : 0.375461
  idt_b               : 0.176579
  Total training time: 0.16 hours


## Inference

Cells below let you run G_A (diseased → healthy) and G_B (healthy → diseased) on individual test images.
Load from any saved checkpoint before running inference.

In [ ]:
# 8. Load a specific checkpoint for inference
INFER_CHECKPOINT = os.path.join(CHECKPOINT_DIR, f'cyclegan_epoch{NUM_EPOCHS}_final.pth')

ckpt = torch.load(INFER_CHECKPOINT, map_location=device, weights_only=False)
netG_A.load_state_dict(ckpt['netG_A_state_dict'])
netG_B.load_state_dict(ckpt['netG_B_state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
del ckpt

# List test images for both domains
print(f'\nTest H: {len(test_H)} images')
for i, (_, f) in enumerate(test_H):
    print(f'  [{i}] {f}')
print(f'\nTest D: {len(test_D)} images')
for i, (_, f) in enumerate(test_D):
    print(f'  [{i}] {f}')

In [22]:
# 9. Visualise G_A (diseased → healthy) on a test diseased image
TEST_D_INDEX = 0   # <-- change this

torch.cuda.empty_cache()
netG_A.eval()
real_D_t, fname_d = test_D[TEST_D_INDEX]

with torch.no_grad():
    fake_H_t = netG_A(real_D_t.unsqueeze(0).to(device))
    rec_D_t  = netG_B(fake_H_t)   # backward cycle

def t2np(t):
    return t.squeeze(0).cpu().permute(1, 2, 0).clamp(0, 1).numpy()

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(t2np(real_D_t)); axs[0].set_title(f'Real Diseased\n{fname_d}'); axs[0].axis('off')
axs[1].imshow(t2np(fake_H_t)); axs[1].set_title('G_A: Diseased → Healthy (inpainted)'); axs[1].axis('off')
axs[2].imshow(t2np(rec_D_t));  axs[2].set_title('G_B(G_A(x)): Cycle Reconstruction'); axs[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, f'infer_GA_test{TEST_D_INDEX}.png'), dpi=150, bbox_inches='tight')
plt.show()

# Cycle reconstruction L1
l1_cycle = torch.mean(torch.abs(rec_D_t.cpu() - real_D_t)).item()
print(f'Backward cycle L1 (rec_D vs real_D): {l1_cycle:.4f}')
del fake_H_t, rec_D_t

Backward cycle L1 (rec_D vs real_D): 0.0318


In [25]:
# 10. Visualise G_B (healthy → diseased) on a test healthy image
TEST_H_INDEX = 1   # <-- change this

torch.cuda.empty_cache()
netG_B.eval()
real_H_t, fname_h = test_H[TEST_H_INDEX]

with torch.no_grad():
    fake_D_t = netG_B(real_H_t.unsqueeze(0).to(device))
    rec_H_t  = netG_A(fake_D_t)   # forward cycle

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(t2np(real_H_t)); axs[0].set_title(f'Real Healthy\n{fname_h}'); axs[0].axis('off')
axs[1].imshow(t2np(fake_D_t)); axs[1].set_title('G_B: Healthy → Diseased (synthetic GLS)'); axs[1].axis('off')
axs[2].imshow(t2np(rec_H_t));  axs[2].set_title('G_A(G_B(x)): Cycle Reconstruction'); axs[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, f'infer_GB_test{TEST_H_INDEX}.png'), dpi=150, bbox_inches='tight')
plt.show()

l1_cycle = torch.mean(torch.abs(rec_H_t.cpu() - real_H_t)).item()
print(f'Forward cycle L1 (rec_H vs real_H): {l1_cycle:.4f}')
del fake_D_t, rec_H_t

Forward cycle L1 (rec_H vs real_H): 0.0818


In [26]:
# 11. Full test-set cycle consistency evaluation
torch.cuda.empty_cache()
netG_A.eval()
netG_B.eval()

test_loader_H = DataLoader(test_H, batch_size=1, shuffle=False, num_workers=2)
test_loader_D = DataLoader(test_D, batch_size=1, shuffle=False, num_workers=2)

fwd_l1 = fwd_ssim = 0.0   # forward cycle  (H→D→H)
bwd_l1 = bwd_ssim = 0.0   # backward cycle (D→H→D)
n_H = n_D = 0

with torch.no_grad():
    for (real_H, _) in tqdm(test_loader_H, desc='Forward cycle (H→D→H)'):
        real_H = real_H.to(device)
        fake_D = netG_B(real_H)
        rec_H  = netG_A(fake_D)
        fwd_l1   += criterion_l1(rec_H, real_H).item()
        fwd_ssim += criterion_ssim.ssim_value(rec_H, real_H).item()
        n_H += 1
        del fake_D, rec_H

    for (real_D, _) in tqdm(test_loader_D, desc='Backward cycle (D→H→D)'):
        real_D = real_D.to(device)
        fake_H = netG_A(real_D)
        rec_D  = netG_B(fake_H)
        bwd_l1   += criterion_l1(rec_D, real_D).item()
        bwd_ssim += criterion_ssim.ssim_value(rec_D, real_D).item()
        n_D += 1
        del fake_H, rec_D

print('=' * 55)
print('FULL TEST SET — CYCLE CONSISTENCY')
print('=' * 55)
print(f'  Forward  (H→D→H) — L1: {fwd_l1/n_H:.4f}  SSIM: {fwd_ssim/n_H:.4f}')
print(f'  Backward (D→H→D) — L1: {bwd_l1/n_D:.4f}  SSIM: {bwd_ssim/n_D:.4f}')
print('=' * 55)

Backward cycle (D→H→D): 100%|██████████| 27/27 [00:00<00:00, 34.00it/s]

FULL TEST SET — CYCLE CONSISTENCY
  Forward  (H→D→H) — L1: 0.0951  SSIM: 0.8203
  Backward (D→H→D) — L1: 0.0542  SSIM: 0.8136


In [ ]:
# 12. Evaluate FULL test set (CycleGAN) - average + per-image L1, SSIM, PSNR
import csv

torch.cuda.empty_cache()
netG_A.eval()
netG_B.eval()

# Build loaders locally so this cell can run independently of Cell 11
test_loader_H = DataLoader(test_H, batch_size=1, shuffle=False, num_workers=2)
test_loader_D = DataLoader(test_D, batch_size=1, shuffle=False, num_workers=2)

fwd_l1 = fwd_ssim = fwd_psnr = 0.0   # H -> D -> H
bwd_l1 = bwd_ssim = bwd_psnr = 0.0   # D -> H -> D
n_H = n_D = 0

# Per-image metrics (for statistical testing)
fwd_metrics = []  # [{'direction', 'filename', 'l1', 'ssim', 'psnr'}]
bwd_metrics = []

with torch.no_grad():
    # Forward cycle: healthy -> fake diseased -> reconstructed healthy
    for real_H, fname_H in tqdm(test_loader_H, desc='Forward cycle metrics (H->D->H)'):
        real_H = real_H.to(device, non_blocking=True)
        fake_D = netG_B(real_H)
        rec_H = netG_A(fake_D)

        l1_val = criterion_l1(rec_H, real_H).item()
        ssim_val = criterion_ssim.ssim_value(rec_H, real_H).item()
        mse_val = torch.mean((rec_H - real_H) ** 2).item()
        psnr_val = 10 * np.log10(1.0 / (mse_val + 1e-10))

        fwd_l1 += l1_val
        fwd_ssim += ssim_val
        fwd_psnr += psnr_val

        fwd_metrics.append({
            'direction': 'H->D->H',
            'filename': fname_H[0],
            'l1': l1_val,
            'ssim': ssim_val,
            'psnr': psnr_val,
        })

        n_H += 1
        del fake_D, rec_H

    # Backward cycle: diseased -> fake healthy -> reconstructed diseased
    for real_D, fname_D in tqdm(test_loader_D, desc='Backward cycle metrics (D->H->D)'):
        real_D = real_D.to(device, non_blocking=True)
        fake_H = netG_A(real_D)
        rec_D = netG_B(fake_H)

        l1_val = criterion_l1(rec_D, real_D).item()
        ssim_val = criterion_ssim.ssim_value(rec_D, real_D).item()
        mse_val = torch.mean((rec_D - real_D) ** 2).item()
        psnr_val = 10 * np.log10(1.0 / (mse_val + 1e-10))

        bwd_l1 += l1_val
        bwd_ssim += ssim_val
        bwd_psnr += psnr_val

        bwd_metrics.append({
            'direction': 'D->H->D',
            'filename': fname_D[0],
            'l1': l1_val,
            'ssim': ssim_val,
            'psnr': psnr_val,
        })

        n_D += 1
        del fake_H, rec_D

avg_fwd_l1 = fwd_l1 / max(n_H, 1)
avg_fwd_ssim = fwd_ssim / max(n_H, 1)
avg_fwd_psnr = fwd_psnr / max(n_H, 1)

avg_bwd_l1 = bwd_l1 / max(n_D, 1)
avg_bwd_ssim = bwd_ssim / max(n_D, 1)
avg_bwd_psnr = bwd_psnr / max(n_D, 1)

# Arrays ready for t-tests
fwd_ssim_scores = np.array([r['ssim'] for r in fwd_metrics], dtype=np.float64)
bwd_ssim_scores = np.array([r['ssim'] for r in bwd_metrics], dtype=np.float64)
fwd_psnr_scores = np.array([r['psnr'] for r in fwd_metrics], dtype=np.float64)
bwd_psnr_scores = np.array([r['psnr'] for r in bwd_metrics], dtype=np.float64)

print('=' * 60)
print('FULL TEST SET EVALUATION (CYCLE RECONSTRUCTION METRICS)')
print('=' * 60)
print(f'  Forward  (H->D->H): L1={avg_fwd_l1:.6f}  SSIM={avg_fwd_ssim:.6f}  PSNR={avg_fwd_psnr:.2f} dB')
print(f'  Backward (D->H->D): L1={avg_bwd_l1:.6f}  SSIM={avg_bwd_ssim:.6f}  PSNR={avg_bwd_psnr:.2f} dB')

if n_H > 0 and n_D > 0:
    mean_l1 = 0.5 * (avg_fwd_l1 + avg_bwd_l1)
    mean_ssim = 0.5 * (avg_fwd_ssim + avg_bwd_ssim)
    mean_psnr = 0.5 * (avg_fwd_psnr + avg_bwd_psnr)
    print('-' * 60)
    print(f'  Mean (both cycles): L1={mean_l1:.6f}  SSIM={mean_ssim:.6f}  PSNR={mean_psnr:.2f} dB')
print('=' * 60)

# Save per-image metrics for external stats workflows
per_image_rows = fwd_metrics + bwd_metrics
metrics_csv = os.path.join(LOG_DIR, 'cyclegan_test_per_image_metrics.csv')
with open(metrics_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['direction', 'filename', 'l1', 'ssim', 'psnr'])
    writer.writeheader()
    writer.writerows(per_image_rows)

print(f'Per-image metrics saved: {metrics_csv}')
print(f'Forward samples: {len(fwd_ssim_scores)} | Backward samples: {len(bwd_ssim_scores)}')
print('Variables available for t-tests: fwd_ssim_scores, bwd_ssim_scores, fwd_psnr_scores, bwd_psnr_scores')